# Uncovered Interest Rate Parity (UIP): USA vs. Euro Area
## Empirical Analysis Based on Quarterly Data 2022–2025

---

**Course:** Fundamentals of Exchange Rate Theory and Exchange Rate Markets  
**Level:** Bachelor Economics

---

## How to Use This Notebook

You do **not** need to write any code. Your task is to **understand and interpret** each step.

**Run a cell:** Click on the cell, then press **Shift + Enter**  
**All code is hidden by default.** To inspect the R code behind any output, click the small **▶ ···** toggle on the left side of the cell.

In [ ]:
# Setup
required_packages <- c("dplyr","ggplot2","lubridate","tidyr","scales","knitr")
for (pkg in required_packages) {
  if (!require(pkg, character.only=TRUE, quietly=TRUE)) {
    install.packages(pkg, repos="https://cran.rstudio.com/", quiet=TRUE)
    library(pkg, character.only=TRUE, quietly=TRUE)
  }
}
options(repr.plot.width=10, repr.plot.height=5)

In [ ]:
# Load data
us_raw <- read.csv("DTB3.csv"); colnames(us_raw) <- c("date","value")
eu_raw <- read.csv("IR3TIB01DEM156N.csv"); colnames(eu_raw) <- c("date","value")
fx_raw <- read.csv("DEXUSEU.csv"); colnames(fx_raw) <- c("date","value")
clean <- function(df) df |>
  dplyr::mutate(date=as.Date(date), value=suppressWarnings(as.numeric(value))) |>
  dplyr::filter(!is.na(value))
us_raw <- clean(us_raw); eu_raw <- clean(eu_raw); fx_raw <- clean(fx_raw)

In [ ]:
# Quarterly aggregation
start_date <- as.Date("2022-01-01"); end_date <- as.Date("2025-12-31")
to_quarterly <- function(df, varname) {
  df |> filter(date>=start_date, date<=end_date) |>
    mutate(month_start=floor_date(date,"month")) |>
    group_by(month_start) |>
    summarise(value=mean(value,na.rm=TRUE),.groups="drop") |>
    mutate(qstart=floor_date(month_start,"quarter")) |>
    filter(month_start==qstart) |> select(qstart,value) |>
    rename(!!varname:=value)
}
us_q <- to_quarterly(us_raw,"i_US")
eu_q <- to_quarterly(eu_raw,"i_EU")
fx_q <- to_quarterly(fx_raw,"S_start")
fx_end_raw <- fx_raw |>
  filter(date>=start_date, date<=end_date+months(3)) |>
  mutate(month_start=floor_date(date,"month")) |>
  group_by(month_start) |>
  summarise(value=mean(value,na.rm=TRUE),.groups="drop") |>
  mutate(monthnum=as.integer(format(month_start,"%m")),
         qstart=floor_date(month_start,"quarter")) |>
  filter(monthnum %in% c(1,4,7,10)) |>
  mutate(qstart=qstart-months(3)) |>
  filter(qstart>=start_date) |> select(qstart,value) |> rename(S_end=value)
df <- us_q |> inner_join(eu_q,by="qstart") |>
  inner_join(fx_q,by="qstart") |> inner_join(fx_end_raw,by="qstart") |>
  filter(!is.na(i_US),!is.na(i_EU),!is.na(S_start),!is.na(S_end)) |>
  mutate(
    zins_diff      = i_US - i_EU,
    S_UIP          = S_start*(1+i_US/400)/(1+i_EU/400),
    delta_s_actual = (S_end/S_start-1)*100,
    delta_s_uip    = (S_UIP/S_start-1)*100,
    abweichung     = delta_s_actual - delta_s_uip,
    abweichung_pct = (S_end/S_UIP-1)*100
  )

---

## 1. Interactive UIP Explorer

Before diving into the data, let's build an **intuition** for the UIP.

The UIP states that the expected exchange rate change over the **next three months** must exactly offset the interest rate differential. The sliders show **annualised** interest rates, but the prediction refers to the **quarterly change** (3 months):

$$S_1 = S_0 \cdot \frac{1 + i_{US}/400}{1 + i_{EU}/400}$$

**Use the sliders** to explore how the UIP-implied exchange rate responds to interest rate shifts.

In [ ]:
library(IRdisplay)
html_code <- '
<style>
.uip-box{font-family:Calibri,Arial,sans-serif;max-width:700px;background:#f8f9fa;
  border:1px solid #dee2e6;border-radius:8px;padding:24px;margin:10px 0}
.uip-title{font-size:17px;font-weight:bold;margin-bottom:6px;color:#212529}
.uip-sub{font-size:13px;color:#6c757d;margin-bottom:16px}
.slider-row{display:flex;align-items:center;margin:10px 0;gap:12px}
.slider-label{width:270px;font-size:14px;color:#495057}
input[type=range]{flex:1;accent-color:#1f77b4}
.slider-val{width:55px;text-align:right;font-weight:bold;font-size:14px;color:#1f77b4}
.result-box{margin-top:20px;background:white;border-left:4px solid #1f77b4;padding:14px 18px;border-radius:4px}
.result-row{display:flex;justify-content:space-between;margin:6px 0;font-size:14px}
.result-label{color:#495057}.result-val{font-weight:bold}
.pos{color:#d62728}.neg{color:#2ca02c}.neu{color:#333}
.interp{margin-top:14px;padding:10px 14px;border-radius:4px;font-size:14px;line-height:1.6}
.dep{background:#fff3cd;border:1px solid #ffc107}
.app{background:#d4edda;border:1px solid #28a745}
.par{background:#e2e3e5;border:1px solid #adb5bd}
.note{font-size:18px;font-family:Calibri,Arial,sans-serif;color:#6c757d;margin-top:12px;font-style:italic}
</style>
<div class="uip-box">
<div class="uip-title">UIP Interactive Explorer</div>
<div class="uip-sub">Anchor: Q1 2022 &mdash; S0 = 1.1317 USD/EUR &nbsp;|&nbsp; Sliders: annualised rates &nbsp;|&nbsp; Prediction: 3-month horizon</div>
<div class="slider-row">
  <span class="slider-label">US Interest Rate i_US (% p.a., annualised)</span>
  <input type="range" id="iUS" min="-2" max="8" step="0.25" value="0.15" oninput="calcUIP()">
  <span class="slider-val" id="vUS">0.15</span>
</div>
<div class="slider-row">
  <span class="slider-label">Euro Area Interest Rate i_EU (% p.a., annualised)</span>
  <input type="range" id="iEU" min="-2" max="8" step="0.25" value="-0.56" oninput="calcUIP()">
  <span class="slider-val" id="vEU">-0.56</span>
</div>
<div class="result-box">
  <div class="result-row"><span class="result-label">Spot Rate S0 (USD/EUR)</span><span class="result-val neu">1.1317</span></div>
  <div class="result-row"><span class="result-label">Interest Differential i_US minus i_EU (annualised)</span><span class="result-val" id="rDiff">--</span></div>
  <div class="result-row"><span class="result-label">Quarterly differential (= annual / 4)</span><span class="result-val" id="rDiffQ">--</span></div>
  <div class="result-row"><span class="result-label">UIP-Implied Rate S1 in 3 months (USD/EUR)</span><span class="result-val" id="rS1">--</span></div>
  <div class="result-row"><span class="result-label">Predicted USD Change over 3 months</span><span class="result-val" id="rChg">--</span></div>
</div>
<div class="interp par" id="interp">Adjust the sliders to see the UIP prediction.</div>
<div class="note">Note: the predicted exchange rate change is always much smaller than the annualised interest differential, because it applies only to a single quarter (3 months = 1/4 of a year).</div>
</div>
<script>
function calcUIP(){
  var S0=1.1317;
  var iUS=parseFloat(document.getElementById("iUS").value);
  var iEU=parseFloat(document.getElementById("iEU").value);
  document.getElementById("vUS").textContent=iUS.toFixed(2);
  document.getElementById("vEU").textContent=iEU.toFixed(2);
  var S1=S0*(1+iUS/400)/(1+iEU/400);
  var diff=iUS-iEU; var diffQ=diff/4; var chg=(S1/S0-1)*100;
  document.getElementById("rDiff").textContent=diff.toFixed(2)+" pp";
  document.getElementById("rDiff").className="result-val"+(diff>0.05?" pos":diff<-0.05?" neg":" neu");
  document.getElementById("rDiffQ").textContent=diffQ.toFixed(3)+" pp";
  document.getElementById("rS1").textContent=S1.toFixed(4)+" USD/EUR";
  var sign=(chg>=0)?"+":"";
  document.getElementById("rChg").textContent=sign+chg.toFixed(3)+"%";
  document.getElementById("rChg").className="result-val"+(chg>0.001?" pos":chg<-0.001?" neg":" neu");
  var box=document.getElementById("interp");
  if(Math.abs(diff)<0.1){
    box.className="interp par";
    box.innerHTML="Interest rates are equal &mdash; UIP predicts <strong>no exchange rate change</strong> over the next 3 months.";
  } else if(diff>0){
    box.className="interp dep";
    box.innerHTML="US rates are higher by <strong>"+diff.toFixed(2)+" pp</strong> (annualised). Over the next <strong>3 months</strong>, UIP predicts a USD <strong>depreciation</strong> of "+Math.abs(chg).toFixed(3)+"% &mdash; just enough to offset the quarterly interest advantage of "+(diff/4).toFixed(3)+" pp.";
  } else {
    box.className="interp app";
    box.innerHTML="Euro Area rates are higher by <strong>"+Math.abs(diff).toFixed(2)+" pp</strong> (annualised). Over the next <strong>3 months</strong>, UIP predicts a USD <strong>appreciation</strong> of "+Math.abs(chg).toFixed(3)+"%.";
  }
}
calcUIP();
</script>
'
IRdisplay::display_html(html_code)

---

## 2. Theoretical Background

### The Uncovered Interest Rate Parity (UIP)

Imagine you have €1,000 and can choose between two investments:

**Option A – Invest in the Euro Area:**

$$\text{€}1{,}000 \cdot \left(1 + \frac{i_{EU}}{400}\right)$$

**Option B – Invest in the USA:**

Convert to USD at spot rate $S_t$, invest at the US rate, convert back after 3 months:

$$\text{€}1{,}000 \cdot S_t \cdot \left(1 + \frac{i_{US}}{400}\right) \cdot \frac{1}{S_{t+1}}$$

**The UIP condition** — both options yield the same return in equilibrium:

$$\boxed{S_{t+1}^{\text{UIP}} = S_t \cdot \frac{1 + i_{US}/400}{1 + i_{EU}/400}}$$

> **Note on the divisor 400:** FRED interest rates are annualised percentages. Dividing by 100 converts to decimals; dividing by 4 gives the **quarterly** rate — hence 400.

### Data Sources

| Variable | FRED Code | Description |
|---|---|---|
| $i_{US}$ | `DTB3` | 3-Month Treasury Bill Rate, USA |
| $i_{EU}$ | `IR3TIB01DEM156N` | 3-Month Interbank Rate, Germany (Euro Area proxy) |
| $S_t$ | `DEXUSEU` | USD per EUR, Spot Exchange Rate |

---

## 3. Interest Rate Developments: USA vs. Euro Area

The chart below shows the 3-month interest rates for the USA and the Euro Area from Q1 2022 to Q4 2025.

In [ ]:
df |>
  select(qstart,i_US,i_EU) |>
  pivot_longer(c(i_US,i_EU),names_to="Region",values_to="Rate") |>
  mutate(Region=recode(Region,
    i_US="USA (3M Treasury Bill)",
    i_EU="Euro Area (3M Interbank DE)")) |>
ggplot(aes(x=qstart,y=Rate,color=Region,linetype=Region)) +
  geom_line(linewidth=1.3) + geom_point(size=3) +
  scale_color_manual(values=c(
    "USA (3M Treasury Bill)"="dodgerblue3",
    "Euro Area (3M Interbank DE)"="firebrick3")) +
  scale_x_date(date_breaks="6 months",date_labels="%b %Y") +
  labs(title="3-Month Interest Rates: USA vs. Euro Area (2022-2025)",
       subtitle="Beginning-of-quarter values | Source: FRED",
       x=NULL,y="Interest Rate (% p.a.)",color=NULL,linetype=NULL) +
  theme_minimal(base_size=13) +
  theme(legend.position="bottom",
        axis.text.x=element_text(angle=30,hjust=1),
        panel.grid.minor=element_blank())

In [ ]:
library(IRdisplay)
q1 <- '
<style>
.qbox{font-family:Calibri,Arial,sans-serif;max-width:700px;background:#f0f4ff;
  border:1px solid #b0c4de;border-radius:8px;padding:22px;margin:12px 0}
.qtitle{font-size:16px;font-weight:bold;color:#1a3a6b;margin-bottom:16px}
.qblock{margin-bottom:18px;padding-bottom:14px;border-bottom:1px solid #ccd6f0}
.qblock:last-child{border-bottom:none;margin-bottom:0}
.qtext{font-size:14px;font-weight:bold;color:#212529;margin-bottom:8px}
.qopts label{display:block;margin:5px 0;font-size:14px;cursor:pointer;padding:4px 8px;border-radius:4px}
.qopts label:hover{background:#dce6ff}
.fb{margin-top:8px;padding:8px 12px;border-radius:4px;font-size:13px;display:none}
.ok{background:#d4edda;color:#155724;border:1px solid #c3e6cb}
.no{background:#f8d7da;color:#721c24;border:1px solid #f5c6cb}
.btn{margin-top:8px;padding:6px 16px;background:#1f77b4;color:white;
  border:none;border-radius:4px;cursor:pointer;font-size:13px}
.btn:hover{background:#1560a0}
</style>
<div class="qbox">
<div class="qtitle">&#128214; Task 1 — Quick Check: Interest Rate Developments</div>
<div class="qblock">
<div class="qtext">a) In which period were US rates most significantly higher than Euro Area rates?</div>
<div class="qopts">
  <label><input type="radio" name="q1" value="a"> 2022 Q1 — both near zero</label>
  <label><input type="radio" name="q1" value="b"> 2022 Q3 to 2024 Q1 — Fed tightened faster than ECB</label>
  <label><input type="radio" name="q1" value="c"> 2025 Q3 — rates converged</label>
</div>
<div class="fb" id="f1"></div>
<button class="btn" onclick="chk(1)">Check</button>
</div>
<div class="qblock">
<div class="qtext">b) According to UIP: if US rates are higher, what should happen to the USD?</div>
<div class="qopts">
  <label><input type="radio" name="q2" value="a"> USD appreciates — investors prefer higher yield</label>
  <label><input type="radio" name="q2" value="b"> USD depreciates — interest advantage is offset by exchange rate loss</label>
  <label><input type="radio" name="q2" value="c"> USD unchanged — UIP relates rates only to inflation</label>
</div>
<div class="fb" id="f2"></div>
<button class="btn" onclick="chk(2)">Check</button>
</div>
<div class="qblock">
<div class="qtext">c) When do US and Euro Area rates converge most noticeably?</div>
<div class="qopts">
  <label><input type="radio" name="q3" value="a"> Early 2022 — both start near zero</label>
  <label><input type="radio" name="q3" value="b"> Mid 2023 — ECB catches up</label>
  <label><input type="radio" name="q3" value="c"> Late 2024/2025 — both central banks cut rates</label>
</div>
<div class="fb" id="f3"></div>
<button class="btn" onclick="chk(3)">Check</button>
</div>
</div>
<script>
var ans={"1":"b","2":"b","3":"c"};
var exp={
  "1":{"b":"Correct! From ~Q3 2022 the Fed tightened aggressively while the ECB lagged, creating a large differential peaking around 2023.",
       "x":"Not quite. The largest differential emerged as the Fed raised rates much faster than the ECB, roughly Q3 2022 to Q1 2024."},
  "2":{"b":"Correct! UIP says the interest advantage must be exactly offset by USD depreciation — otherwise arbitrage profits would exist.",
       "x":"Not quite. UIP predicts the higher-yielding currency must depreciate to eliminate the arbitrage opportunity."},
  "3":{"c":"Correct! By late 2024 and into 2025, both the Fed and ECB began cutting rates, causing convergence.",
       "x":"Look again at the chart. The most notable convergence happens at the end of the sample as both central banks shifted to rate cuts."}
};
function chk(n){
  var sel=document.querySelector("input[name=q"+n+"]:checked");
  var fb=document.getElementById("f"+n);
  fb.style.display="block";
  if(!sel){fb.className="fb no";fb.textContent="Please select an answer.";return;}
  if(sel.value===ans[n]){fb.className="fb ok";fb.textContent="✓ "+exp[n][sel.value];}
  else{fb.className="fb no";fb.textContent="✗ "+exp[n]["x"];}
}
</script>
'
IRdisplay::display_html(q1)

---

## 4. UIP Forecast vs. Actual Exchange Rate

The chart below compares the **UIP forecast** with the **actual realised exchange rate** at end-of-quarter (lines, left axis, USD per EUR, scaled 0.8–1.2). The right axis shows the **percentage deviation** of the actual rate from the UIP forecast.

In [ ]:
mid <- mean(c(df$S_UIP, df$S_end))
scale_r <- 0.05

ggplot(df, aes(x=qstart)) +
  geom_col(aes(y = abweichung_pct * scale_r + mid,
               fill="Deviation actual vs UIP (right axis)"),
           width=60, alpha=0.5, show.legend=TRUE) +
  geom_line(aes(y=S_UIP,color="UIP Forecast",linetype="UIP Forecast"),linewidth=1.2) +
  geom_point(aes(y=S_UIP,color="UIP Forecast"),shape=17,size=3.5) +
  geom_line(aes(y=S_end,color="Actual Rate",linetype="Actual Rate"),linewidth=1.2) +
  geom_point(aes(y=S_end,color="Actual Rate"),shape=16,size=3.5) +
  scale_color_manual(values=c("UIP Forecast"="#2ca02c","Actual Rate"="#d62728")) +
  scale_linetype_manual(values=c("UIP Forecast"="dashed","Actual Rate"="solid")) +
  scale_fill_manual(values=c("Deviation actual vs UIP (right axis)"="#9ecae1")) +
  scale_x_date(date_breaks="6 months",date_labels="%b %Y") +
  scale_y_continuous(
    name   = "USD per EUR",
    limits = c(0.8, 1.2),
    sec.axis = sec_axis(
      trans = ~ (. - mid) / scale_r,
      name  = "Deviation: Actual minus UIP Forecast (%)"
    )
  ) +
  labs(
    title    = "UIP Forecast vs. Actual USD/EUR Exchange Rate",
    subtitle = "Lines: exchange rates (left, 0.8-1.2) | Bars: % deviation from UIP forecast (right) | Source: FRED",
    x=NULL, color=NULL, linetype=NULL, fill=NULL
  ) +
  theme_minimal(base_size=13) +
  theme(legend.position="bottom",
        axis.text.x=element_text(angle=30,hjust=1),
        panel.grid.minor=element_blank(),
        axis.title.y.right=element_text(color="#4a90d9"),
        axis.text.y.right=element_text(color="#4a90d9"))

**Reading this chart:** The green dashed line shows what the exchange rate *should have been* according to UIP; the red solid line shows what *actually happened*. The blue bars on the right axis measure the percentage gap between actual and forecast. Large bars indicate quarters where the UIP prediction was far off. The systematic presence of large deviations is the core empirical challenge to the UIP.

In [ ]:
library(IRdisplay)
q2 <- '
<style>
.qbox{font-family:Calibri,Arial,sans-serif;max-width:700px;background:#f0f4ff;
  border:1px solid #b0c4de;border-radius:8px;padding:22px;margin:12px 0}
.qtitle{font-size:16px;font-weight:bold;color:#1a3a6b;margin-bottom:16px}
.qblock{margin-bottom:18px;padding-bottom:14px;border-bottom:1px solid #ccd6f0}
.qblock:last-child{border-bottom:none;margin-bottom:0}
.qtext{font-size:14px;font-weight:bold;color:#212529;margin-bottom:8px}
.qopts label{display:block;margin:5px 0;font-size:14px;cursor:pointer;padding:4px 8px;border-radius:4px}
.qopts label:hover{background:#dce6ff}
.fb{margin-top:8px;padding:8px 12px;border-radius:4px;font-size:13px;display:none}
.ok{background:#d4edda;color:#155724;border:1px solid #c3e6cb}
.no{background:#f8d7da;color:#721c24;border:1px solid #f5c6cb}
.btn{margin-top:8px;padding:6px 16px;background:#1f77b4;color:white;border:none;border-radius:4px;cursor:pointer;font-size:13px}
.btn:hover{background:#1560a0}
</style>
<div class="qbox">
<div class="qtitle">&#128214; Task 2 — Quick Check: UIP Forecast vs. Reality</div>
<div class="qblock">
<div class="qtext">a) In Q3 2022, the actual rate (red) is well below the UIP forecast (green). What does this mean for the USD?</div>
<div class="qopts">
  <label><input type="radio" name="q21" value="a"> USD depreciated more than UIP predicted</label>
  <label><input type="radio" name="q21" value="b"> USD depreciated less than UIP predicted (or even appreciated)</label>
  <label><input type="radio" name="q21" value="c"> USD and UIP forecast coincided exactly</label>
</div>
<div class="fb" id="f21"></div>
<button class="btn" onclick="chk2(1)">Check</button>
</div>
<div class="qblock">
<div class="qtext">b) Looking at the blue deviation bars: are the deviations randomly positive and negative, or do you see a pattern?</div>
<div class="qopts">
  <label><input type="radio" name="q22" value="a"> Mostly random — no clear pattern</label>
  <label><input type="radio" name="q22" value="b"> Mostly positive in 2022–2023 when the US differential was largest</label>
  <label><input type="radio" name="q22" value="c"> Perfectly symmetric around zero each year</label>
</div>
<div class="fb" id="f22"></div>
<button class="btn" onclick="chk2(2)">Check</button>
</div>
<div class="qblock">
<div class="qtext">c) Overall, is UIP a reliable short-run predictor of exchange rate movements in this sample?</div>
<div class="qopts">
  <label><input type="radio" name="q23" value="a"> Yes — the green and red lines track each other closely</label>
  <label><input type="radio" name="q23" value="b"> No — deviations are large and systematic, suggesting other forces dominate</label>
  <label><input type="radio" name="q23" value="c"> Only in 2024–2025 when rates converged</label>
</div>
<div class="fb" id="f23"></div>
<button class="btn" onclick="chk2(3)">Check</button>
</div>
</div>
<script>
var ans2={"1":"b","2":"b","3":"b"};
var exp2={
  "1":{"b":"Correct! If the actual rate is below the forecast (fewer USD per EUR), the USD was stronger than UIP predicted — it depreciated less, or even appreciated.",
       "x":"Look at the axis: fewer USD per EUR means the USD was stronger, not weaker, than the UIP prediction."},
  "2":{"b":"Correct! Large positive deviations cluster in 2022-2023, when the interest differential was at its peak — suggesting a systematic failure of UIP in high-differential environments.",
       "x":"Look more carefully at the bars: they are not symmetric. Large positive deviations cluster when the US-EU differential was largest."},
  "3":{"b":"Correct! The large, often systematic deviations confirm that short-run exchange rate dynamics are driven by forces beyond interest differentials.",
       "x":"The lines diverge substantially in most quarters — the UIP is not a reliable short-run predictor in this sample."}
};
function chk2(n){
  var sel=document.querySelector("input[name=q2"+n+"]:checked");
  var fb=document.getElementById("f2"+n);
  fb.style.display="block";
  if(!sel){fb.className="fb no";fb.textContent="Please select an answer.";return;}
  if(sel.value===ans2[n]){fb.className="fb ok";fb.textContent="✓ "+exp2[n][sel.value];}
  else{fb.className="fb no";fb.textContent="✗ "+exp2[n]["x"];}
}
</script>
'
IRdisplay::display_html(q2)

---

## 5. Excess Volatility: Interest Differential vs. Exchange Rate Changes

A key feature of foreign exchange markets is **excess volatility**: actual exchange rate changes are far larger in magnitude than what the interest rate differential predicts.

In [ ]:
max_fx    <- max(abs(df$delta_s_actual))
max_zdiff <- max(abs(df$zins_diff))
scale_f   <- max_zdiff / max_fx

ggplot(df, aes(x=qstart)) +
  geom_col(aes(y=zins_diff,fill="Interest Differential (left axis)"),
           width=60,alpha=0.7) +
  geom_line(aes(y=delta_s_actual*scale_f,
                color="Actual FX Change (right axis)"),linewidth=1.3) +
  geom_point(aes(y=delta_s_actual*scale_f,
                 color="Actual FX Change (right axis)"),size=3) +
  geom_hline(yintercept=0,linewidth=0.6) +
  scale_fill_manual(values=c("Interest Differential (left axis)"="#aec7e8")) +
  scale_color_manual(values=c("Actual FX Change (right axis)"="#d62728")) +
  scale_x_date(date_breaks="6 months",date_labels="%b %Y") +
  scale_y_continuous(
    name = "Interest Rate Differential i_US - i_EU (pp, annualised)",
    sec.axis = sec_axis(
      trans=~./scale_f,
      name="Actual USD/EUR Change over quarter (%)")) +
  labs(title="Excess Volatility: Interest Differential vs. Actual Exchange Rate Change",
       subtitle="Bars: interest differential at quarter start (left) | Line: quarterly FX change (right) | Source: FRED",
       x=NULL,fill=NULL,color=NULL) +
  theme_minimal(base_size=13) +
  theme(legend.position="bottom",
        axis.text.x=element_text(angle=30,hjust=1),
        panel.grid.minor=element_blank(),
        axis.title.y.right=element_text(color="#d62728"),
        axis.text.y.right=element_text(color="#d62728"))

**Reading this chart:** The blue bars show the interest differential at the start of each quarter — this is the sole input to the UIP forecast. The red line shows the actual exchange rate change. Notice that the red line swings far beyond what the bars would suggest: exchange rate fluctuations are typically **five to ten times larger** than the interest differential. This excess volatility means that risk appetite, news shocks, and capital flow dynamics far outweigh the interest rate channel in driving short-run exchange rates.

In [ ]:
library(IRdisplay)
q3 <- '
<style>
.qbox{font-family:Calibri,Arial,sans-serif;max-width:700px;background:#f0f4ff;
  border:1px solid #b0c4de;border-radius:8px;padding:22px;margin:12px 0}
.qtitle{font-size:16px;font-weight:bold;color:#1a3a6b;margin-bottom:16px}
.qblock{margin-bottom:18px;padding-bottom:14px;border-bottom:1px solid #ccd6f0}
.qblock:last-child{border-bottom:none;margin-bottom:0}
.qtext{font-size:14px;font-weight:bold;color:#212529;margin-bottom:8px}
.qopts label{display:block;margin:5px 0;font-size:14px;cursor:pointer;padding:4px 8px;border-radius:4px}
.qopts label:hover{background:#dce6ff}
.fb{margin-top:8px;padding:8px 12px;border-radius:4px;font-size:13px;display:none}
.ok{background:#d4edda;color:#155724;border:1px solid #c3e6cb}
.no{background:#f8d7da;color:#721c24;border:1px solid #f5c6cb}
.btn{margin-top:8px;padding:6px 16px;background:#1f77b4;color:white;border:none;border-radius:4px;cursor:pointer;font-size:13px}
.btn:hover{background:#1560a0}
</style>
<div class="qbox">
<div class="qtitle">&#128214; Task 3 — Quick Check: Excess Volatility</div>
<div class="qblock">
<div class="qtext">a) The interest differential (bars) stays in a narrow range. By roughly how much do the actual FX changes (line) exceed the differential in magnitude?</div>
<div class="qopts">
  <label><input type="radio" name="q31" value="a"> About the same magnitude</label>
  <label><input type="radio" name="q31" value="b"> About 2–3 times larger</label>
  <label><input type="radio" name="q31" value="c"> Roughly 5–10 times larger</label>
</div>
<div class="fb" id="f31"></div>
<button class="btn" onclick="chk3(1)">Check</button>
</div>
<div class="qblock">
<div class="qtext">b) Are the exchange rate changes at least consistent in direction with the interest differential?</div>
<div class="qopts">
  <label><input type="radio" name="q32" value="a"> Yes — when the US differential is positive, the USD always depreciates</label>
  <label><input type="radio" name="q32" value="b"> No — the direction of FX changes is largely unrelated to the sign of the differential</label>
  <label><input type="radio" name="q32" value="c"> Only in years when inflation was above target</label>
</div>
<div class="fb" id="f32"></div>
<button class="btn" onclick="chk3(2)">Check</button>
</div>
<div class="qblock">
<div class="qtext">c) What does excess volatility imply for using UIP as a short-run forecasting tool?</div>
<div class="qopts">
  <label><input type="radio" name="q33" value="a"> UIP is highly reliable — small errors only</label>
  <label><input type="radio" name="q33" value="b"> UIP provides a useful direction signal but misses most of the magnitude</label>
  <label><input type="radio" name="q33" value="c"> UIP performs poorly — other factors dominate short-run exchange rate dynamics</label>
</div>
<div class="fb" id="f33"></div>
<button class="btn" onclick="chk3(3)">Check</button>
</div>
</div>
<script>
var ans3={"1":"c","2":"b","3":"c"};
var exp3={
  "1":{"c":"Correct! Compare the right axis (FX changes up to ~8%) with the left axis (differential rarely above 2 pp). The ratio is roughly 5-10x.",
       "x":"Look at the two axes carefully. The right axis (red line) reaches values several times larger than the left axis (blue bars)."},
  "2":{"b":"Correct! The red line changes sign multiple times even when the differential stays positive throughout — direction is largely unpredictable from the differential alone.",
       "x":"Look at 2022-2023: the differential is persistently positive, yet the FX line swings both positive and negative with no clear pattern."},
  "3":{"c":"Correct! Because the unexplained volatility far exceeds the interest channel, UIP is a poor short-run predictor — risk premia and news shocks dominate.",
       "x":"Given the massive excess volatility, UIP provides very limited short-run forecasting power — the errors are not small."}
};
function chk3(n){
  var sel=document.querySelector("input[name=q3"+n+"]:checked");
  var fb=document.getElementById("f3"+n);
  fb.style.display="block";
  if(!sel){fb.className="fb no";fb.textContent="Please select an answer.";return;}
  if(sel.value===ans3[n]){fb.className="fb ok";fb.textContent="✓ "+exp3[n][sel.value];}
  else{fb.className="fb no";fb.textContent="✗ "+exp3[n]["x"];}
}
</script>
'
IRdisplay::display_html(q3)

---

## 6. Statistical UIP Test: Regression Analysis

To formally test the UIP, economists estimate the following regression:

$$\Delta s_{t+1} = \alpha + \beta \cdot (i_{US,t} - i_{EU,t}) + \varepsilon_{t+1}$$

where $\Delta s_{t+1} = \left(\frac{S_{t+1}}{S_t} - 1\right) \times 100$ is the actual percentage change in the USD/EUR exchange rate over the quarter.

**Interpretation and empirical context:**  
(1) The UIP predicts $\alpha = 0$ and $\beta = 0.25$: a one percentage point higher annualised US rate implies a 0.25% quarterly USD depreciation.  
(2) If $\hat{\beta} \approx 0$, the interest differential has no predictive power for exchange rate changes — the UIP fails to hold in the data.  
(3) A **negative** $\hat{\beta}$ implies that high-interest currencies actually *appreciate* rather than depreciate — the exact opposite of UIP, known as the **Forward Premium Puzzle**.  
(4) The typically very low $R^2$ confirms that short-run exchange rate dynamics are dominated by risk premia, capital flow shocks, and market sentiment — not interest differentials.

In [ ]:
reg <- lm(delta_s_actual ~ zins_diff, data=df)
summary(reg)

In [ ]:
beta_hat <- coef(reg)[2]
ggplot(df,aes(x=zins_diff,y=delta_s_actual)) +
  geom_point(size=3.5,color="#1f77b4",alpha=0.85) +
  geom_text(aes(label=paste0(year(qstart)," Q",quarter(qstart))),
            vjust=-0.9,size=3,color="#333333") +
  geom_smooth(method="lm",se=TRUE,color="#d62728",fill="#f7b6b6",linewidth=1.2) +
  geom_abline(intercept=0,slope=0.25,linetype="dashed",color="#2ca02c",linewidth=1.2) +
  annotate("text",x=Inf,y=Inf,label=paste0("OLS: beta = ",round(beta_hat,3)),
           hjust=1.1,vjust=1.8,size=4.5,color="#d62728") +
  annotate("text",x=Inf,y=-Inf,label="UIP prediction: beta = 0.25",
           hjust=1.1,vjust=-0.8,size=4.5,color="#2ca02c") +
  labs(title="UIP Test: Interest Rate Differential vs. Actual Exchange Rate Change",
       subtitle="Red: OLS estimate | Green dashed: UIP prediction (beta=0.25) | Source: FRED",
       x="Interest Rate Differential i_US - i_EU (pp, annualised)",
       y="Actual Exchange Rate Change (%, USD/EUR)") +
  theme_minimal(base_size=13) + theme(panel.grid.minor=element_blank())

In [ ]:
library(IRdisplay)
q4 <- '
<style>
.qbox{font-family:Calibri,Arial,sans-serif;max-width:700px;background:#f0f4ff;
  border:1px solid #b0c4de;border-radius:8px;padding:22px;margin:12px 0}
.qtitle{font-size:16px;font-weight:bold;color:#1a3a6b;margin-bottom:16px}
.qblock{margin-bottom:18px;padding-bottom:14px;border-bottom:1px solid #ccd6f0}
.qblock:last-child{border-bottom:none;margin-bottom:0}
.qtext{font-size:14px;font-weight:bold;color:#212529;margin-bottom:8px}
.qopts label{display:block;margin:5px 0;font-size:14px;cursor:pointer;padding:4px 8px;border-radius:4px}
.qopts label:hover{background:#dce6ff}
.fb{margin-top:8px;padding:8px 12px;border-radius:4px;font-size:13px;display:none}
.ok{background:#d4edda;color:#155724;border:1px solid #c3e6cb}
.no{background:#f8d7da;color:#721c24;border:1px solid #f5c6cb}
.btn{margin-top:8px;padding:6px 16px;background:#1f77b4;color:white;border:none;border-radius:4px;cursor:pointer;font-size:13px}
.btn:hover{background:#1560a0}
</style>
<div class="qbox">
<div class="qtitle">&#128214; Task 4 — Quick Check: Regression Results</div>
<div class="qblock">
<div class="qtext">a) The OLS estimate of beta is shown in red on the scatter plot. Is it significantly different from zero, and in which direction?</div>
<div class="qopts">
  <label><input type="radio" name="q41" value="a"> Significantly positive — consistent with UIP</label>
  <label><input type="radio" name="q41" value="b"> Not significantly different from zero — no predictive power</label>
  <label><input type="radio" name="q41" value="c"> The sign and significance depend entirely on the sample period</label>
</div>
<div class="fb" id="f41"></div>
<button class="btn" onclick="chk4(1)">Check</button>
</div>
<div class="qblock">
<div class="qtext">b) What would a negative beta coefficient imply about the relationship between interest differentials and exchange rate changes?</div>
<div class="qopts">
  <label><input type="radio" name="q42" value="a"> High-interest currencies depreciate as UIP predicts</label>
  <label><input type="radio" name="q42" value="b"> High-interest currencies tend to appreciate — the Forward Premium Puzzle</label>
  <label><input type="radio" name="q42" value="c"> Interest rates have no effect on exchange rates</label>
</div>
<div class="fb" id="f42"></div>
<button class="btn" onclick="chk4(2)">Check</button>
</div>
<div class="qblock">
<div class="qtext">c) What does a very low R-squared (e.g. below 0.10) tell us about the UIP regression?</div>
<div class="qopts">
  <label><input type="radio" name="q43" value="a"> The interest differential explains most of the exchange rate variation</label>
  <label><input type="radio" name="q43" value="b"> The interest differential explains only a tiny fraction of exchange rate movements</label>
  <label><input type="radio" name="q43" value="c"> The regression model is incorrectly specified</label>
</div>
<div class="fb" id="f43"></div>
<button class="btn" onclick="chk4(3)">Check</button>
</div>
</div>
<script>
var ans4={"1":"c","2":"b","3":"b"};
var exp4={
  "1":{"c":"Correct! With only 16 observations, results are sensitive to the sample. In many samples beta is not significant or even negative — consistent with the Forward Premium Puzzle literature.",
       "x":"With a short sample of 16 quarters, statistical significance is hard to establish. The key finding in the broader literature is that beta is often negative or insignificant."},
  "2":{"b":"Correct! This is the Forward Premium Puzzle: the currency with the higher interest rate tends to appreciate rather than depreciate, directly contradicting UIP.",
       "x":"A negative beta means high-interest currencies tend to appreciate — the opposite of UIP. This is the Forward Premium Puzzle."},
  "3":{"b":"Correct! A low R-squared means the interest differential accounts for very little of the variation in exchange rate changes — other factors dominate.",
       "x":"R-squared measures the share of variation explained by the model. A very low value means the interest differential has little explanatory power."}
};
function chk4(n){
  var sel=document.querySelector("input[name=q4"+n+"]:checked");
  var fb=document.getElementById("f4"+n);
  fb.style.display="block";
  if(!sel){fb.className="fb no";fb.textContent="Please select an answer.";return;}
  if(sel.value===ans4[n]){fb.className="fb ok";fb.textContent="✓ "+exp4[n][sel.value];}
  else{fb.className="fb no";fb.textContent="✗ "+exp4[n]["x"];}
}
</script>
'
IRdisplay::display_html(q4)

---

## 7. Summary

| Concept | Key Message |
|---|---|
| **UIP Formula** | $S_{t+1}^{UIP} = S_t \cdot \frac{1+i_{US}/400}{1+i_{EU}/400}$ |
| **Interest Differential** | USA had significantly higher rates than the Euro Area in 2022–2023 |
| **UIP Forecast** | According to UIP, the USD should have depreciated substantially |
| **Reality** | Actual movements deviated considerably from UIP predictions |
| **Excess Volatility** | FX changes are far larger than the interest differential predicts |
| **Forward Premium Puzzle** | High-interest currencies tend to appreciate empirically — opposite of UIP |

---

### Task 5 — Final Reflection

In **3–5 sentences**: What do the results imply for an investor who tries to profit from interest differentials? What risks does the UIP framework overlook?

**Your answer to Task 5:**

...

---
*Data source: Federal Reserve Bank of St. Louis (FRED) — fred.stlouisfed.org*  
*Series: DTB3, IR3TIB01DEM156N, DEXUSEU*